#  Fine-Tuning Qwen 2.5-Coder 7B with QLoRA on Google Colab

**A Production-Ready Tutorial for Supervised Fine-Tuning (SFT) on a T4 GPU**

---

This notebook walks you through the complete pipeline of fine-tuning the
`Qwen/Qwen2.5-Coder-7B-Instruct` model using **QLoRA** (Quantized Low-Rank Adaptation)
on a free-tier Google Colab T4 GPU (≈15 GB VRAM).

| Phase | Description |
|-------|-------------|
| **Phase 1** | Environment Setup — Drive, secrets, library installs |
| **Phase 2** | Model & Tokenizer Initialization — 4-bit quantization + LoRA |
| **Phase 3** | Dataset Preparation — Loading, formatting (ChatML), tokenization |
| **Phase 4** | Training Configuration — `TrainingArguments` + `SFTTrainer` |
| **Phase 5** | Inference & Testing — Generate, evaluate, save, and push |


---
# Phase 1: Environment Setup

Before we begin training, we need to:
1. **Mount Google Drive** — to persist model checkpoints across Colab sessions.
2. **Authenticate with Hugging Face** — many models (including Qwen) require an access token.
3. **Install required libraries** — the fine-tuning stack (`transformers`, `peft`, `trl`, etc.).


## 1.1 Mount Google Drive

### 📖 Guide
Google Colab instances are **ephemeral** — all files are lost when the runtime disconnects.
Mounting Google Drive gives us a persistent storage location where we can save:
- Training checkpoints (so we can resume interrupted training)
- The final LoRA adapter weights
- The merged full-precision model


In [ ]:
# Mount Google Drive to persist checkpoints and model artifacts
from google.colab import drive  # Colab-specific module for Drive integration
drive.mount('/content/drive')   # Mounts your entire Google Drive at this path

## 1.2 Authenticate with Hugging Face Hub

### 📖 Guide
Many models on the Hugging Face Hub are **gated** — they require you to accept a license
and use an access token to download weights. Even for public models, authenticating
lets you push your fine-tuned model back to the Hub.

> **Security Best Practice:** We use `userdata.get('HF_TOKEN')` to read the token from
> Colab's **Secrets** tab (🔑 icon in the sidebar). This avoids hard-coding tokens in code
> that might be shared or committed to Git.


In [ ]:
from google.colab import userdata       # Colab API to read secrets securely
from huggingface_hub import login        # Official HF Hub login function

# Retrieve the token stored in Colab Secrets under the key 'HF_TOKEN'
hf_token = userdata.get('HF_TOKEN')

# Authenticate the current session with Hugging Face
login(token=hf_token)

## 1.3 Install Required Libraries

### 📖 Guide
We install the core fine-tuning stack in a single command for speed:

| Library | Purpose |
|---------|:--------|
| `transformers` | Model loading, tokenization, training utilities |
| `datasets` | Efficient loading and pre-processing of HF datasets |
| `peft` | Parameter-Efficient Fine-Tuning — provides LoRA/QLoRA |
| `trl` | `SFTTrainer` — a high-level trainer for instruction tuning |
| `bitsandbytes` | 4-bit / 8-bit quantization kernels |
| `accelerate` | Multi-GPU / mixed-precision training orchestration |
| `flash-attn` | FlashAttention-2 — fused attention kernel for 2–4× speedup |

<br>
> **Why one `pip install` call?** Resolving dependencies once is faster than<br>
> running separate `pip install` commands sequentially.


In [ ]:
# Install all fine-tuning dependencies in one shot for faster resolution
!pip install -q \
    transformers \
    datasets \
    peft \
    trl \
    bitsandbytes \
    accelerate \
    flash-attn --no-build-isolation

## 1.4 Import Libraries

### 📖 Guide
We organize imports into logical groups:
1. **Core** — PyTorch and OS utilities
2. **Data** — Hugging Face `datasets`
3. **Model & Tokenizer** — `transformers` classes for Causal LM
4. **PEFT** — LoRA configuration from the `peft` library
5. **Trainer** — `SFTTrainer` from `trl` for supervised fine-tuning


In [ ]:
# --- 1. Core System & Logic ---
import torch                           # Deep learning framework — backbone for GPU computation
import os                              # Operating system utilities for paths and env vars

# --- 2. Dataset Management ---
from datasets import load_dataset      # Load datasets from Hugging Face Hub or local disk

# --- 3. Model & Tokenizer ---
from transformers import (
    AutoTokenizer,                     # Auto-selects the correct tokenizer for any model
    AutoModelForCausalLM,              # Auto-selects the correct causal LM architecture
    BitsAndBytesConfig,                # Configures 4-bit / 8-bit quantization
    TrainingArguments,                 # Defines hyperparameters (LR, batch size, etc.)
    pipeline,                          # High-level inference API for quick testing
    logging,                           # Controls transformer library log verbosity
)

# --- 4. PEFT (Parameter-Efficient Fine-Tuning) ---
from peft import LoraConfig, PeftModel # LoRA adapter config and model wrapper

# --- 5. Supervised Fine-Tuning Trainer ---
from trl import SFTTrainer             # High-level trainer that handles chat formatting

---
# Phase 2: Model & Tokenizer Initialization

In this phase we:
1. Define the base model identifier.
2. Configure **4-bit quantization** via `BitsAndBytesConfig`.
3. Load the model and tokenizer.
4. Set up the **LoRA** adapter configuration.

### Why QLoRA?
The Qwen 2.5-Coder 7B model has ~7.6 billion parameters. In full `float16`,
that requires **≈15 GB** just for the weights — leaving zero room for optimizer states,
gradients, or activations on a T4's 15 GB VRAM.

**QLoRA** solves this by:
- **Quantizing** the base model to 4-bit (≈3.5 GB for weights)
- **Freezing** all original weights
- **Injecting** tiny trainable LoRA adapters (< 1% of total parameters)


## 2.1 Define the Base Model

### 📖 Guide
We use `Qwen/Qwen2.5-Coder-7B-Instruct` — a 7-billion-parameter model
pre-trained on code and instruction data. The `-Instruct` variant is already
chat-tuned, making it an excellent starting point for domain-specific fine-tuning.


In [ ]:
# The Hugging Face model ID — this uniquely identifies the model on the Hub
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"

## 2.2 Configure 4-Bit Quantization

### 📖 Guide
`BitsAndBytesConfig` controls how the model weights are compressed:

| Parameter | Value | Why |
|-----------|-------|-----|
| `load_in_4bit` | `True` | Compresses each weight from 16 bits → 4 bits (≈4× memory saving) |
| `bnb_4bit_quant_type` | `"nf4"` | *NormalFloat4* — information-theoretically optimal for normally-distributed weights |
| `bnb_4bit_compute_dtype` | `float16` | Computations are done in half-precision for speed |
| `bnb_4bit_use_double_quant` | `True` | Quantizes the quantization constants — saves ~0.4 bits/param extra |


In [ ]:
# Configure 4-bit quantization for memory-efficient model loading
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Enable 4-bit quantization
    bnb_4bit_quant_type="nf4",              # Use NormalFloat4 — optimal for pre-trained weights
    bnb_4bit_compute_dtype=torch.float16,   # Run matrix multiplications in float16
    bnb_4bit_use_double_quant=True,          # Apply nested quantization for extra savings
)

## 2.3 Load the Base Model

### 📖 Guide
We load the model with our quantization config applied. Key parameters:
- `device_map="auto"` automatically distributes layers across available GPUs/CPU.
- `trust_remote_code=True` is needed because Qwen uses custom modeling code.
- `torch_dtype=torch.float16` sets the default dtype for any non-quantized parameters.

> ⚡ This step downloads ~4 GB of quantized weights from the Hub.


In [ ]:
# Load the base model with 4-bit quantization applied
model = AutoModelForCausalLM.from_pretrained(
    model_id,                              # Hugging Face model identifier
    quantization_config=bnb_config,        # Apply our 4-bit quantization settings
    device_map="auto",                     # Auto-place layers on GPU/CPU
    trust_remote_code=True,                # Required for Qwen's custom architecture code
    torch_dtype=torch.float16,             # Default dtype for non-quantized params
)

## 2.4 Load and Configure the Tokenizer

### 📖 Guide
The tokenizer converts raw text into token IDs that the model understands.

Important setup:
- **`padding_side="right"`** — for causal LMs, right-padding is standard so the
  model generates from left to right without confusion.
- **`pad_token = eos_token`** — Qwen doesn't define a separate pad token; we reuse EOS.
- **ChatML special tokens** (`<|im_start|>`, `<|im_end|>`) — we ensure these are
  registered so the tokenizer treats them as single tokens, not character sequences.
- **`resize_token_embeddings`** — after adding new tokens, the model's embedding
  matrix must be resized to accommodate them.


In [ ]:
# Load the tokenizer matching the base model
tokenizer = AutoTokenizer.from_pretrained(
    model_id,                              # Must match the model for correct vocabulary
    trust_remote_code=True,                # Qwen uses a custom tokenizer implementation
    padding_side="right",                  # Right-padding is standard for causal LMs
)

# Set the padding token to the end-of-sequence token (Qwen has no default pad token)
tokenizer.pad_token = tokenizer.eos_token

# Register ChatML special tokens so they are treated as atomic units
special_tokens = ["<|im_start|>", "<|im_end|>"]
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

# Resize the model's embedding layer to include the newly added tokens
model.resize_token_embeddings(len(tokenizer))

## 2.5 Configure LoRA Adapters

### 📖 Guide
**LoRA (Low-Rank Adaptation)** freezes the original model weights and injects
small trainable matrices into specific layers. This lets us fine-tune with
**< 1% of the total parameters**, drastically reducing memory and compute.

| Parameter | Value | Explanation |
|-----------|-------|-------------|
| `r` | `8` | Rank of the low-rank decomposition — higher = more capacity, more VRAM |
| `lora_alpha` | `32` | Scaling factor (effective LR multiplier = α/r = 4) |
| `target_modules` | *see below* | Which linear layers to inject adapters into |
| `lora_dropout` | `0.05` | Light dropout for regularization |
| `bias` | `"none"` | Don't train bias terms — saves memory with minimal quality loss |

**Target modules** for Qwen 2.5 include both attention projections (`q/k/v/o_proj`)
and feed-forward layers (`gate/up/down_proj`) for comprehensive adaptation.


In [ ]:
# Configure LoRA — the parameter-efficient adapter that we actually train
peft_config = LoraConfig(
    r=8,                                   # Low-rank dimension (trade-off: quality vs. VRAM)
    lora_alpha=32,                         # Scaling factor (effective multiplier = alpha/r)
    target_modules=[                       # Layers to inject LoRA adapters into
        "q_proj",                          # Query projection in self-attention
        "k_proj",                          # Key projection in self-attention
        "v_proj",                          # Value projection in self-attention
        "o_proj",                          # Output projection in self-attention
        "gate_proj",                       # Gating layer in the MLP block
        "up_proj",                         # Up-projection in the MLP block
        "down_proj",                       # Down-projection in the MLP block
    ],
    lora_dropout=0.05,                     # Dropout on LoRA layers for regularization
    bias="none",                           # Don't train bias terms (saves memory)
    task_type="CAUSAL_LM",                 # Specifies this is a causal language model task
)

---
# Phase 3: Dataset Preparation

In this phase we:
1. Load the instruction dataset from the Hugging Face Hub.
2. Format each example into the **ChatML** template that Qwen expects.
3. Apply the formatting across the entire dataset.

### Why ChatML?
Qwen 2.5 was pre-trained using the **ChatML** format:
```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>
```
Using the same format during fine-tuning ensures the model leverages its
pre-trained chat capabilities rather than fighting against them.


## 3.1 Load the Dataset

### 📖 Guide
We use **Magicoder-OSS-Instruct-75K** — a curated dataset of 75,000 high-quality
coding instruction-response pairs. Each example has `prompt` and `response` fields.

Shuffling prevents the model from learning spurious patterns based on data ordering.

> **Tip:** When testing your pipeline for the first time, uncomment
> `dataset.select(range(1000))` to do a quick sanity-check run with a tiny subset.


In [ ]:
# Load the 75K coding instruction-response dataset
dataset = load_dataset(
    "ise-uiuc/Magicoder-OSS-Instruct-75K",  # Dataset identifier on Hugging Face Hub
    split="train",                            # Use the training split
)

# Shuffle to prevent order-dependent learning artifacts
dataset = dataset.shuffle(seed=42)

# [OPTIONAL] Use a small subset for a fast sanity check
# dataset = dataset.select(range(1000))

## 3.2 Inspect the Dataset Structure

### 📖 Guide
Always inspect one sample before formatting to understand column names and data types.


In [ ]:
# Preview the first example to check column names and content
dataset[0]

## 3.3 Format Data for ChatML

### 📖 Guide
We define a formatting function that wraps each sample in the ChatML template.
The function uses `.get()` for safe field access — if a field is missing,
it defaults to an empty string instead of crashing.

The resulting `"text"` column is what `SFTTrainer` will use for training.


In [ ]:
def format_prompt(example):
    """Convert a dataset example into Qwen's ChatML format."""
    # Define a system prompt to guide the model's persona
    system_prompt = "You are a helpful assistant."

    # Safely extract fields (using .get() prevents KeyError on missing columns)
    instruction = example.get("prompt", "")    # The user's instruction/question
    input_text = example.get("input", "")       # Optional additional context
    output_text = example.get("response", "")   # The target/expected response

    # Combine instruction and input into a single user query
    user_query = f"{instruction}\n{input_text}".strip()

    # Build the ChatML-formatted training example
    prompt = (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{user_query}<|im_end|>\n"
        f"<|im_start|>assistant\n{output_text}<|im_end|>"
    )

    # Return as a dict — datasets .map() will add this as a new column
    return {"text": prompt}

## 3.4 Apply Formatting to the Entire Dataset

### 📖 Guide
`.map()` applies our formatting function to every row efficiently.
After this step, each example has a `"text"` column containing the full ChatML string.


In [ ]:
# Apply the ChatML formatting function to every example in the dataset
dataset = dataset.map(format_prompt)

---
# Phase 4: Training Configuration

This phase sets up:
1. **`TrainingArguments`** — all hyperparameters (learning rate, batch size, precision, etc.)
2. **`SFTTrainer`** — a high-level trainer from TRL that orchestrates the training loop.

### Memory Budget for a T4 GPU (15 GB VRAM)
| Component | Approx. VRAM |
|-----------|-------------|
| 4-bit model weights | ~3.5 GB |
| LoRA adapter parameters | ~0.1 GB |
| Optimizer states (paged AdamW) | ~0.5 GB |
| Gradients + activations (with checkpointing) | ~8 GB |
| **Total** | **~12 GB** ✅ |


## 4.1 Define Training Arguments

### 📖 Guide
Key choices explained:
- **`per_device_train_batch_size=1`** with **`gradient_accumulation_steps=8`** →
  effective batch size of 8, fitting within T4 memory.
- **`gradient_checkpointing=True`** — trades compute for memory by recomputing activations
  during the backward pass instead of storing them.
- **`optim="paged_adamw_32bit"`** — bitsandbytes paged optimizer that automatically
  offloads optimizer states to CPU when GPU memory is scarce.
- **`bf16=True`** — uses bfloat16 mixed precision; set to `False` and `fp16=True` if
  your GPU doesn't support BF16 (most T4s do via software emulation).
- **`cosine` LR scheduler** with warmup gradually increases then decays the learning rate.


In [ ]:
# Define all training hyperparameters
training_args = TrainingArguments(
    output_dir="./results",                # Directory for checkpoints and logs
    per_device_train_batch_size=1,         # Micro-batch size per GPU (keep at 1 for T4)
    gradient_accumulation_steps=8,         # Accumulate 8 steps → effective batch size = 8
    learning_rate=1e-6,                    # Conservative LR to avoid catastrophic forgetting
    num_train_epochs=1,                    # Number of full passes through the dataset
    max_steps=5400,                        # Hard cap on training steps (overrides epochs)
    logging_steps=100,                     # Log metrics every 100 steps

    # --- Precision & Memory ---
    bf16=True,                             # Use bfloat16 mixed precision (set fp16=True if unsupported)
    fp16=False,                            # Mutually exclusive with bf16
    gradient_checkpointing=True,           # Recompute activations to save ~40% VRAM

    # --- Optimizer & Scheduler ---
    optim="paged_adamw_32bit",             # Paged optimizer — auto-offloads states to CPU
    lr_scheduler_type="cosine",            # Cosine annealing for smooth LR decay
    warmup_ratio=0.1,                      # Warm up for the first 10% of steps
    weight_decay=0.1,                      # L2 regularization to prevent overfitting

    # --- Efficiency ---
    group_by_length=True,                  # Group similar-length sequences to reduce padding
    report_to="tensorboard",               # Log metrics to TensorBoard

    # --- Checkpointing ---
    save_strategy="steps",                 # Save checkpoints at regular step intervals
    save_steps=100,                        # Save every 100 steps
    save_total_limit=2,                    # Keep only the 2 most recent checkpoints
)

## 4.2 Initialize the SFTTrainer

### 📖 Guide
`SFTTrainer` from HuggingFace's TRL library extends the standard `Trainer`
with features specifically designed for instruction tuning:
- Automatically handles the `"text"` column from our dataset.
- Integrates LoRA config so adapter layers are injected transparently.
- Manages tokenization, padding, and data collation internally.


In [ ]:
# Initialize the Supervised Fine-Tuning trainer
trainer = SFTTrainer(
    model=model,                           # The quantized base model
    train_dataset=dataset,                 # Our ChatML-formatted dataset
    peft_config=peft_config,               # LoRA adapter configuration
    args=training_args,                    # Training hyperparameters defined above
)

## 4.3 Start Training

### 📖 Guide
This kicks off the training loop. On a T4 with the settings above,
expect ~1–2 samples/second. Monitor the loss in the output — it should
decrease steadily over the first few hundred steps.


In [ ]:
# Launch the training loop
trainer.train()

## 4.4 Resume Training from Checkpoint (If Interrupted)

### 📖 Guide
Colab sessions can disconnect unexpectedly. If that happens, re-run all cells
above, then use this cell to **resume from the latest checkpoint** in `output_dir`.
The trainer automatically detects the most recent `checkpoint-XXX` folder.


In [ ]:
# Resume training from the most recent checkpoint in './results'
# trainer.train(resume_from_checkpoint=True)

---
# Phase 5: Inference, Evaluation & Deployment

After training completes, we:
1. Run a **quick inference test** on the fine-tuned model.
2. **Save** the LoRA adapter and tokenizer.
3. **Merge** the adapter with the base model (optional — requires more RAM).
4. **Analyze** the model (parameter count, memory footprint).
5. **Evaluate** using perplexity.
6. **Push** to Hugging Face Hub.


## 5.1 Quick Inference Test

### 📖 Guide
Before saving, run a quick generation to verify the model produces
reasonable output. We use the `pipeline` API for simplicity.


In [ ]:
# Create a text generation pipeline with the fine-tuned model
test_pipe = pipeline(
    "text-generation",                     # Task type
    model=trainer.model,                   # Use the model currently in the trainer
    tokenizer=tokenizer,                   # Must match the model's tokenizer
)

# Define a test prompt in ChatML format
prompt = "<|im_start|>user\nWrite a Python function to check if a number is prime.<|im_end|>\n<|im_start|>assistant\n"

# Generate a response with controlled randomness
outputs = test_pipe(
    prompt,
    max_new_tokens=200,                    # Limit response length
    do_sample=True,                        # Enable stochastic sampling
    temperature=0.7,                       # Balance creativity vs. determinism
)

# Display the full generated text
print(outputs[0]['generated_text'])

## 5.2 Save the LoRA Adapter & Tokenizer

### 📖 Guide
We save only the **LoRA adapter weights** (a few hundred MB), not the full 7B model.
This is the key advantage of PEFT — your fine-tuning result is small and portable.
The tokenizer must also be saved because we added special tokens.


In [ ]:
# Define a name for the fine-tuned adapter
new_model_name = "qwen2.5-7b-coding-adapter"

# Save the LoRA adapter weights (only the small delta weights, not the full model)
trainer.model.save_pretrained(new_model_name)

# Save the tokenizer (important because we added ChatML special tokens)
tokenizer.save_pretrained(new_model_name)

## 5.3 Merge LoRA Adapter with Base Model (Optional)

### 📖 Guide
Merging combines the LoRA weights back into the base model, producing a single
standalone model that doesn't require the `peft` library for inference.

> ⚠️ **Memory Warning:** This loads the base model in full precision (`bfloat16`
> ≈15 GB). On a T4, use `device_map="cpu"` to offload to system RAM.


In [ ]:
# Load the base model in full precision for merging (uses CPU RAM)
base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    model_id,                              # Same base model we fine-tuned
    torch_dtype=torch.bfloat16,            # Full bfloat16 precision (no quantization)
    device_map="cpu",                      # Use CPU RAM — T4 VRAM is insufficient
    trust_remote_code=True,                # Required for Qwen
)

# Load the LoRA adapter on top of the base model
merged_model = PeftModel.from_pretrained(base_model_for_merge, new_model_name)

# Merge adapter weights into the base model and unload the PEFT wrapper
merged_model = merged_model.merge_and_unload()

# Save the fully merged model
save_path = "qwen2.5-7b-merged"
merged_model.save_pretrained(save_path, safe_serialization=True)
tokenizer.save_pretrained(save_path)

## 5.4 Backup to Google Drive

### 📖 Guide
Copy the adapter and/or merged model to Google Drive so they persist
beyond the Colab session lifetime.


In [ ]:
import shutil  # Standard library for high-level file operations

# Copy the LoRA adapter to Google Drive
shutil.copytree(
    new_model_name,                                    # Source: local adapter directory
    f"/content/drive/MyDrive/{new_model_name}",        # Destination: Google Drive
)

# Copy the merged model to Google Drive
shutil.copytree(
    save_path,                                         # Source: local merged model
    f"/content/drive/MyDrive/{save_path}",             # Destination: Google Drive
)

print("✅ Files successfully backed up to Google Drive!")

## 5.5 Parameter Analysis

### 📖 Guide
This analysis reveals the efficiency of LoRA. Typically < 1% of parameters
are trainable, yet the model achieves significant task-specific improvements.


In [ ]:
# Count total parameters (frozen + trainable)
total_params = sum(p.numel() for p in model.parameters())

# Count only trainable parameters (the LoRA adapters)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Display the parameter breakdown
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %:          {100 * trainable_params / total_params:.4f}%")

## 5.6 Model Memory Footprint

### 📖 Guide
This measures the **static** memory footprint — the space occupied by model
weights and buffers on your GPU/CPU. It does not include optimizer states
or activation memory (which are only relevant during training).


In [ ]:
# Calculate parameter memory (weights × bytes per element)
param_size = sum(p.nelement() * p.element_size() for p in model.parameters())

# Calculate buffer memory (non-trainable tensors like BatchNorm running stats)
buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())

# Convert total bytes to megabytes
size_mb = (param_size + buffer_size) / 1024**2

# Display the static memory footprint
print(f"Model memory footprint: {size_mb:.2f} MB")

## 5.7 Compare Base vs. Fine-Tuned Output

### 📖 Guide
We define a reusable `generate()` function and test both the original
base model and the merged fine-tuned model on the same prompt.
This side-by-side comparison highlights the effect of fine-tuning.


In [ ]:
def generate(model, prompt, max_new_tokens=150):
    """Generate text from a model given a prompt string."""
    # Tokenize the prompt and move tensors to the model's device
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Run autoregressive generation
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,      # Cap the response length
        temperature=0.7,                    # Sampling temperature (creativity dial)
        top_p=0.9,                          # Nucleus sampling — top 90% probability mass
        pad_token_id=tokenizer.eos_token_id,# Avoid padding warnings
    )

    # Decode the first (and only) sequence, stripping special tokens
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Define a test prompt in ChatML format
test_prompt = (
    "<|im_start|>user\n"
    "Build a mini project in C++ programming language.\n"
    "<|im_end|>\n"
    "<|im_start|>assistant\n"
)

# Suppress verbose transformer warnings during generation
logging.set_verbosity(logging.CRITICAL)

# --- Base Model Output ---
print("=" * 60)
print("BASE MODEL OUTPUT:")
print("=" * 60)
# Note: base_model_for_merge was loaded in Section 5.3
print(generate(base_model_for_merge, test_prompt))

# --- Fine-Tuned Model Output ---
print("\n" + "=" * 60)
print("FINE-TUNED MODEL OUTPUT:")
print("=" * 60)
print(generate(merged_model, test_prompt))

## 5.8 Evaluate Perplexity

### 📖 Guide
**Perplexity** measures how well the model predicts a sequence.
Lower perplexity = better predictions. It's defined as:

$$PP = e^{\text{avg cross-entropy loss}}$$

We evaluate on a subset of the training data for a quick quality check.


In [ ]:
import math  # For the exponential function in perplexity calculation

def compute_perplexity(model, dataset, num_samples=100):
    """Compute perplexity on a subset of the dataset."""
    model.eval()                           # Set model to evaluation mode (disables dropout)
    losses = []                            # Accumulate per-example losses

    for example in dataset.select(range(num_samples)):
        # Tokenize with truncation to fit model's max context length
        inputs = tokenizer(
            example["text"],
            return_tensors="pt",
            truncation=True,
        ).to(model.device)

        # Disable gradient computation for memory efficiency
        with torch.no_grad():
            # Model auto-computes cross-entropy loss when labels are provided
            outputs = model(**inputs, labels=inputs["input_ids"])

        losses.append(outputs.loss.item()) # Extract scalar loss value

    # Perplexity = exp(average loss)
    return math.exp(sum(losses) / len(losses))


# Compute and display perplexity
ppl = compute_perplexity(merged_model, dataset)
print(f"Perplexity: {ppl:.2f}")

## 5.9 Visualize Training Metrics with TensorBoard

### 📖 Guide
TensorBoard provides interactive plots of your training loss, learning rate,
and other metrics logged during training.


In [ ]:
# Load the TensorBoard extension and point it to the training logs
%load_ext tensorboard
%tensorboard --logdir results/runs

## 5.10 Push to Hugging Face Hub

### 📖 Guide
Upload your fine-tuned model to the Hugging Face Hub for sharing and deployment.
The `check_pr=True` flag creates a Pull Request instead of pushing directly,
which is safer for collaboration.


In [ ]:
import locale
# Fix potential encoding issues in Colab
locale.getpreferredencoding = lambda: "UTF-8"

# Push the merged model and tokenizer to Hugging Face Hub
# Replace 'your-username' with your actual Hugging Face username
hub_model_name = "your-username/qwen2.5-7b-finetuned"  # <-- CHANGE THIS

merged_model.push_to_hub(hub_model_name, check_pr=True)  # Push model weights
tokenizer.push_to_hub(hub_model_name, check_pr=True)     # Push tokenizer

---
# Appendix: Adapting This Pipeline for Other Models

This pipeline works for **any causal LLM**. Three things typically change:

---

### A — LoRA Target Modules
| Model Family | `target_modules` |
|-------------|------------------|
| **Qwen 2.5** | `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj` |
| **LLaMA / TinyLlama** | `q_proj, v_proj` |
| **Mistral** | `q_proj, k_proj, v_proj, o_proj` |
| **Falcon** | `query_key_value` |

---

### B — Prompt Template
| Model | Template |
|-------|----------|
| **Qwen (ChatML)** | `<\|im_start\|>user\n{prompt}<\|im_end\|>` |
| **LLaMA 2 Chat** | `<s>[INST] {prompt} [/INST]` |
| **Alpaca** | `### Instruction:\n{prompt}\n### Response:` |

---

### C — Model Loader Class
| Model Type | Class |
|-----------|-------|
| **Causal LM** (GPT, LLaMA, Qwen, Mistral) | `AutoModelForCausalLM` |
| **Seq2Seq** (T5, FLAN-T5) | `AutoModelForSeq2SeqLM` |
| **Encoder** (BERT) | `AutoModelForSequenceClassification` |
